# Semantic Model TOM Catalog

Connect to a Fabric semantic model through Semantic Link Labs and TOM, then catalog tables, columns, relationships, and measures. The connection is read-only and uses the notebook user's Fabric identity.

## Prerequisites

Run this notebook in a Fabric notebook runtime with access to the target semantic model. The runtime identity must have permission to read the model metadata.

In [ ]:
# Install the Semantic Link Labs package if it is not already available.
%pip install -q semantic-link-labs pandas

In [ ]:
import pandas as pd
import sempy.fabric as fabric
from sempy.fabric.exceptions import FabricHTTPException
from sempy_labs import admin
from sempy_labs.tom import connect_semantic_model

## Configuration

Set the scan scope and write mode here, then run the notebook top to bottom. Leave both filters as `None` to catalog every workspace and model the notebook identity can see. Outputs are written to the **attached lakehouse**, so make sure a lakehouse is attached before running.

In [ ]:
# Scan scope: None means every workspace / model visible to the notebook identity.
WORKSPACE_NAME = None  # Optional exact workspace-name filter.
MODEL_NAME = None  # Optional exact model-name filter.

# Results are written to the lakehouse attached to this notebook.
WRITE_MODE = "overwrite"  # "overwrite" replaces prior output; use "append" only if downstream supports it.

if WORKSPACE_NAME is not None and not WORKSPACE_NAME.strip():
    raise ValueError("WORKSPACE_NAME must be None or a non-empty workspace name.")

## Discover and catalog all workspaces

When `WORKSPACE_NAME` is `None`, the notebook discovers every workspace visible to the Fabric identity, lists its semantic models, and opens a read-only TOM session for each model. Set `WORKSPACE_NAME` or `MODEL_NAME` to narrow the scan.

In [ ]:
try:
    workspaces_df = admin.list_workspaces()
except AttributeError:
    workspaces_df = fabric.list_workspaces()
except FabricHTTPException as error:
    workspaces_df = fabric.list_workspaces()


if workspaces_df.empty:
    raise ValueError("No workspaces are visible to the notebook identity.")

if "Id" in workspaces_df.columns:
    workspace_id_column = "Id"
elif "Workspace Id" in workspaces_df.columns:
    workspace_id_column = "Workspace Id"
else:
    workspace_id_column = "id"

if "Name" in workspaces_df.columns:
    workspace_name_column = "Name"
elif "Workspace Name" in workspaces_df.columns:
    workspace_name_column = "Workspace Name"
else:
    workspace_name_column = "displayName"

if WORKSPACE_NAME is not None:
    workspaces_df = workspaces_df[
        workspaces_df[workspace_name_column].astype(str).str.casefold() == WORKSPACE_NAME.casefold()
    ]

if workspaces_df.empty:
    raise ValueError(f"No matching workspace was found for WORKSPACE_NAME={WORKSPACE_NAME!r}")

datasource_rows = []
table_rows = []
column_rows = []
relationship_rows = []
measure_rows = []
model_rows = []
errors_rows = []

for _, workspace in workspaces_df.iterrows():
    workspace_id = str(workspace[workspace_id_column])
    workspace_name = str(workspace[workspace_name_column])
    print(f"Scanning workspace: {workspace_name}")

    try:
        datasets_df = fabric.list_datasets(workspace=workspace_id)
    except Exception as error:
        errors_rows.append({
            "workspace_id": workspace_id,
            "workspace_name": workspace_name,
            "model_id": None,
            "model_name": None,
            "error_type": type(error).__name__,
            "error_message": f"Could not list semantic models: {error}",
        })
        continue

    if datasets_df.empty:
        continue

    model_id_column = "Dataset ID" if "Dataset ID" in datasets_df.columns else "Id"
    model_name_column = "Dataset Name" if "Dataset Name" in datasets_df.columns else "Name"

    if MODEL_NAME is not None:
        datasets_df = datasets_df[
            datasets_df[model_name_column].astype(str).str.casefold() == MODEL_NAME.casefold()
        ]

    for _, dataset in datasets_df.iterrows():
        model_id = str(dataset[model_id_column])
        model_name = str(dataset[model_name_column])
        print(f"  Extracting model: {model_name}")

        try:
            with connect_semantic_model(
                dataset=model_id,
                workspace=workspace_id,
                readonly=True,
            ) as tom:
                model = tom.model

                model_rows.append({
                    "workspace_id": workspace_id,
                    "workspace_name": workspace_name,
                    "model_id": model_id,
                    "model_name": model_name,
                    "compatibility_level": getattr(model, "CompatibilityLevel", None),
                    "default_mode": str(getattr(model, "DefaultMode", None)) if getattr(model, "DefaultMode", None) is not None else None,
                })

                for datasource in model.DataSources:
                    datasource_rows.append({
                        "workspace_id": workspace_id,
                        "workspace_name": workspace_name,
                        "model_id": model_id,
                        "model_name": model_name,
                        "datasource_name": str(getattr(datasource, "Name", "") or ""),
                        "datasource_type": str(getattr(datasource, "Type", None)) if getattr(datasource, "Type", None) is not None else None,
                        "connection_string": getattr(datasource, "ConnectionString", None),
                        "connection_details": str(getattr(datasource, "ConnectionDetails", None)) if getattr(datasource, "ConnectionDetails", None) is not None else None,
                        "impersonation_mode": str(getattr(datasource, "ImpersonationMode", None)) if getattr(datasource, "ImpersonationMode", None) is not None else None,
                        "description": getattr(datasource, "Description", None),
                    })

                # Shared M expressions carry the real source (SQL endpoint / server + database) for
                # Direct Lake and parameterized Power Query models, where DataSources is empty.
                for expression in getattr(model, "Expressions", []):
                    datasource_rows.append({
                        "workspace_id": workspace_id,
                        "workspace_name": workspace_name,
                        "model_id": model_id,
                        "model_name": model_name,
                        "datasource_name": str(getattr(expression, "Name", "") or ""),
                        "datasource_type": "M expression",
                        "connection_string": getattr(expression, "Expression", None),
                        "connection_details": None,
                        "impersonation_mode": None,
                        "description": getattr(expression, "Description", None),
                    })

                # Inline Power Query partition sources carry the connection for import models.
                # Entity / Direct Lake partitions only reference a shared expression (captured above),
                # so they are skipped here to avoid matching different sources on a generic name.
                for table in model.Tables:
                    for partition in getattr(table, "Partitions", []):
                        partition_source = getattr(partition, "Source", None)
                        m_expression = getattr(partition_source, "Expression", None) if partition_source is not None else None
                        if not m_expression:
                            continue
                        datasource_rows.append({
                            "workspace_id": workspace_id,
                            "workspace_name": workspace_name,
                            "model_id": model_id,
                            "model_name": model_name,
                            "datasource_name": str(getattr(partition, "Name", "") or ""),
                            "datasource_type": type(partition_source).__name__,
                            "connection_string": str(m_expression),
                            "connection_details": None,
                            "impersonation_mode": None,
                            "description": None,
                        })

                for table in model.Tables:
                    table_name = str(getattr(table, "Name", "") or "")
                    table_rows.append({
                        "workspace_id": workspace_id,
                        "workspace_name": workspace_name,
                        "model_id": model_id,
                        "model_name": model_name,
                        "table_name": table_name,
                        "description": getattr(table, "Description", None),
                        "is_hidden": getattr(table, "IsHidden", None),
                    })

                    for column in getattr(table, "Columns", []):
                        column_rows.append({
                            "workspace_id": workspace_id,
                            "workspace_name": workspace_name,
                            "model_id": model_id,
                            "model_name": model_name,
                            "table_name": table_name,
                            "column_name": str(getattr(column, "Name", "") or ""),
                            "data_type": str(getattr(column, "DataType", None)) if getattr(column, "DataType", None) is not None else None,
                            "description": getattr(column, "Description", None),
                            "is_hidden": getattr(column, "IsHidden", None),
                        })

                    for measure in getattr(table, "Measures", []):
                        measure_rows.append({
                            "workspace_id": workspace_id,
                            "workspace_name": workspace_name,
                            "model_id": model_id,
                            "model_name": model_name,
                            "table_name": table_name,
                            "measure_name": str(getattr(measure, "Name", "") or ""),
                            "expression": getattr(measure, "Expression", None),
                            "format_string": getattr(measure, "FormatString", None),
                            "description": getattr(measure, "Description", None),
                            "is_hidden": getattr(measure, "IsHidden", None),
                        })

                for relationship in getattr(model, "Relationships", []):
                    from_table = getattr(relationship, "FromTable", None)
                    from_column = getattr(relationship, "FromColumn", None)
                    to_table = getattr(relationship, "ToTable", None)
                    to_column = getattr(relationship, "ToColumn", None)
                    relationship_rows.append({
                        "workspace_id": workspace_id,
                        "workspace_name": workspace_name,
                        "model_id": model_id,
                        "model_name": model_name,
                        "relationship_name": str(getattr(relationship, "Name", "") or ""),
                        "from_table": str(getattr(from_table, "Name", "") or ""),
                        "from_column": str(getattr(from_column, "Name", "") or ""),
                        "to_table": str(getattr(to_table, "Name", "") or ""),
                        "to_column": str(getattr(to_column, "Name", "") or ""),
                        "cross_filtering_behavior": str(getattr(relationship, "CrossFilteringBehavior", None)) if getattr(relationship, "CrossFilteringBehavior", None) is not None else None,
                        "is_active": getattr(relationship, "IsActive", None),
                    })
        except Exception as error:
            errors_rows.append({
                "workspace_id": workspace_id,
                "workspace_name": workspace_name,
                "model_id": model_id,
                "model_name": model_name,
                "error_type": type(error).__name__,
                "error_message": str(error),
            })

models_df = pd.DataFrame(model_rows)
datasources_df = pd.DataFrame(datasource_rows)
tables_df = pd.DataFrame(table_rows)
columns_df = pd.DataFrame(column_rows)
relationships_df = pd.DataFrame(relationship_rows)
measures_df = pd.DataFrame(measure_rows)
errors_df = pd.DataFrame(errors_rows)

print(f"Cataloged {len(models_df)} models across {len(workspaces_df)} workspaces.")
print(f"Datasources: {len(datasources_df)} | Tables: {len(tables_df)} | Columns: {len(columns_df)} | Relationships: {len(relationships_df)} | Measures: {len(measures_df)}")
if not errors_df.empty:
    print(f"Workspace/model errors: {len(errors_df)}")


## Review and persist the catalog

The catalog is workspace-wide, and each DataFrame includes model context. `errors_df` records models that could not be read so the run still completes for the rest. The next cell previews the results (starting with a per-workspace model count), and the final cell writes them as Delta tables to the attached lakehouse.

In [ ]:
# Per-workspace model counts give a quick overview before the detailed frames.
if not models_df.empty:
    workspace_summary = (
        models_df.groupby("workspace_name")
        .size()
        .reset_index(name="model_count")
        .sort_values("model_count", ascending=False)
        .reset_index(drop=True)
    )
    display(workspace_summary)

display(models_df)
display(datasources_df)
display(tables_df)
display(columns_df)
display(relationships_df)
display(measures_df)
display(errors_df)

In [ ]:
catalog_outputs = {
    "semantic_models": models_df,
    "semantic_model_datasources": datasources_df,
    "semantic_model_tables": tables_df,
    "semantic_model_columns": columns_df,
    "semantic_model_relationships": relationships_df,
    "semantic_model_measures": measures_df,
    "semantic_model_catalog_errors": errors_df,
}

for table_name, frame in catalog_outputs.items():
    if frame.empty:
        print(f"Skipped {table_name}: no rows")
        continue

    spark.createDataFrame(frame).write.format("delta").mode(WRITE_MODE).option(
        "overwriteSchema", "true"
    ).saveAsTable(table_name)
    print(f"Wrote {len(frame)} rows to {table_name}")

## Next steps

The catalog tables are now in the attached lakehouse. Open **`semantic_model_similarity.ipynb`** (attached to this same lakehouse) and run it to score model pairs, flag duplicates, and review the similarity heatmap.